## 1. Install Dependancies

In [19]:
!pip install cohere
!pip install python-dotenv
!pip install rank-bm25

In [4]:
import cohere
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('COHERE_API_KEY')

In [5]:
#assign the api key to the cohere api key
co = cohere.Client(api_key)


In [6]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""


In [7]:
#split into sentences
texts = text.split('.')

#Clean up to remove empty spaces and new lines
texts = [t.strip('\n') for t in texts]


## 2. Embedding the Text Chunks

In [11]:
import numpy as np

#Get embeddings
response = co.embed(
    texts = texts,
    input_type = 'search_document',
).embeddings

embeds = np.array(response)
print(embeds.shape)

(15, 4096)


## 3. Building The Search Index

In [ ]:
import faiss

dim = embeds.shape[1]

#uses the FAISS L2 distance metric (used for vector similarity) to measure the distance between the vectors
index = faiss.IndexFlatL2(dim)

#add the floating point values of the embeddings to the index
index.add(np.float32(embeds))

# 4. Search The Index

### Using Semantic Search

In [14]:
import pandas as pd

def search(query, number_of_results = 5):

    #1. Get tjhe query's embedding
    query_embed = co.embed(
        texts = [query],
        input_type = 'search_query',
    ).embeddings[0]

    #2. Retrieve the nearest neighbours
    distances, similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

    #3. Format the results
    texts_np = np.array(texts)
    results = pd.DataFrame(data = {'texts': texts_np[similar_item_ids[0]],
    'distances': distances[0]})


    #4. Print the results
    print(f"Query:'{query}'\nNearest neighbors:")
    return results



    

In [15]:
#5. Test the search function
query = 'What is the plot of interstellar?'
results = search(query)
print(results)


Query:'What is the plot of interstellar?'
Nearest neighbors:
                                               texts    distances
0  Interstellar is a 2014 epic science fiction fi...  5625.486328
1   Since its premiere, Interstellar gained a cul...  7074.600586
2  It stars Matthew McConaughey, Anne Hathaway, J...  8548.993164
3  Set in a dystopian future where humanity is st...  8601.597656
4  Caltech theoretical physicist and 2017 Nobel l...  8821.138672


In [17]:
#5. Test the search function
query = 'how precise was the science'
results = search(query)
print(results)


Query:'how precise was the science'
Nearest neighbors:
                                               texts     distances
0  It has also received praise from many astronom...  10757.371094
1  Caltech theoretical physicist and 2017 Nobel l...  11566.135742
2  Interstellar uses extensive practical and mini...  11922.841797
3  Cinematographer Hoyte van Hoytema shot it on 3...  12673.580078
4   Since its premiere, Interstellar gained a cul...  13212.718750


### Using Lexical Search

In [20]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc


In [21]:
from tqdm import tqdm

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

100%|██████████| 15/15 [00:00<00:00, 15397.59it/s]


In [22]:
def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

In [23]:
keyword_search(query = "how precise was the science")


Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


# 5. Re-ranking

In [26]:
query = 'how precise was the science'
results = co.rerank(query = query, documents = texts, top_n = 3, return_documents = True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.15239799),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.050354082),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'), index=0, relevance_score=0.0350424)]

In [27]:
for idx, result in enumerate(results.results):
    print(idx, result.relevance_score , result.document.text)


0 0.15239799 It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
1 0.050354082 The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
2 0.0350424 Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan


### pretty darn good ^